# Can an LLM write training data for a BERT model?

### Gwilym Hughes, Summer 2026 Internship

The main reason for looking towards synthetic training data to train AI models is that annotation is the bottleneck to starting new projects. Getting an LLM to write labelled examples instead skips the bottleneck. In theory the knowledge a large LLM has in its weights should be sufficient to generate good training data for a smaller model like BERT, i.e. distillation. This notebook is the result of an 8-week internship trying this for drug-drug interaction extraction.

Data is drawn from the open DDICorpus-2013 in BRAT format and from results generated at various stages of the project: 400 sentences from each of five generator versions, 400 real sentences labelled three ways (by a person, by the model, and by the model after checking its own positive labels), and the scores for the experiments presented in the write-up.

To learn about the core task refer to this earlier notebook, showing a setup for a similar finetuning BERT for DDI-2013 extraction `./sentence_level_relation_extraction_with_BERT.ipynb`

## Setup

DDI-2013 corpus can be downloaded in BRAT format by running the cell below. Other calculated data is sampled or selected from generations and training runs from the internship so no GPU or API key is required.

In [ ]:
!pip install -q bioc spacy scikit-learn pandas matplotlib
!python -c "import spacy; spacy.cli.download('en_core_web_sm')"
![ -d ../DDICorpusBrat ] || (cd .. && curl -sL -o DDICorpus.zip "https://github.com/isegura/DDICorpus/raw/refs/heads/master/DDICorpus-2013(BRAT).zip" && unzip -oq DDICorpus.zip)

In [ ]:
import math, random, re
from collections import Counter, defaultdict

import pandas as pd

import lib
from lib import load_samples, human_sentences, sentence_label, masked, flatten

synth = {v: load_samples(v) for v in lib.VERSIONS}
human = human_sentences("train", n=400, seed=0)
data = {"human": human, **synth}
R = lib.results()
{k: len(v) for k, v in data.items()}

TAGS = re.compile(r"\[(E[12])\](.*?)\[/\1\]")

def pair_names(text):
    found = {t: re.sub(r"\[/?E[12]\]", "", name) for t, name in TAGS.findall(text)}
    return found.get("E1"), found.get("E2")

The generated sentences are 400-sentence samples from the datasets the logged results were measured on, each joined back to the specification that produced it. v13 is the exception because its raw generation output was not kept, so its sentences have no specification. Scores are read from `results.json`, which is rebuilt from the project's run records and keeps the run ids behind each figure. `lib.py` includes helpers for loading, masking and training.

In [ ]:
cols = ["dataset_id", "gen_id", "sentences_available", "sentences_sampled",
        "raw_file_found", "specs_joined"]
man = pd.DataFrame(lib.sample_manifest()).T
display(man[[c for c in cols if c in man.columns]])

v18 = lib.results()["trajectory"]["v18"]
print(f"v18 scored {v18['f1']} (sd {v18['sd']}) over {v18['n_seeds']} seeds, "
      f"from runs {', '.join(v18['run_ids'])}")

## 1. The task

From the BRAT formatted articles we extract the sentences with drugs tagged and listed as pairs of each drug. The BERT model needs to be able to label these pairs, identifying whether there is a stated interaction between the two drugs. Every pair of drugs needs exactly one of five labels: MECHANISM for a pharmacokinetic change, EFFECT for a clinical or pharmacodynamic consequence, ADVISE for a recommendation about taking the two together, INT for an interaction stated without detail, and NONE. Here is an example:

In [ ]:
ex = [r for r in human if sentence_label(r) == "MULT"][0]
print(ex["sentence"], "\n")
pd.DataFrame([{"drug A": a, "drug B": b, "label": p["label"]}
              for p in ex["pairs"] for a, b in [pair_names(p["text"])]])

Because negative relationships are the majority of interaction relationships, five pairs in six in the corpus are NONE. Therefore scores are micro-F1 over the four positive labels to prevent rewarding NONE overprediction.

Every dataset below, generated or real, is held as a list of records of this shape: the sentence, the number of drug mentions, every pair with its label, and, for generated sentences, the specification that produced it. Real sentences are built with the project's pair construction, one instance per unordered pair. The earlier tutorial makes one per ordered pair, whereas here we treat them as unordered because the labels are symmetric as per DDI-2013 guidelines.

In [ ]:
def shape(record):
    def show(v):
        if isinstance(v, list):
            return f"list of {len(v)}"
        if isinstance(v, dict):
            return "specification: " + ", ".join(sorted(v))
        if isinstance(v, str) and len(v) > 60:
            return v[:57] + "..."
        return v
    return {k: show(v) for k, v in record.items()}

pd.DataFrame({"generated (v18)": shape(synth["v18"][0]), "real": shape(human[0])})

## 2. Writing labelled sentences

Labels are given to the generator for it to produce, since this allows class size control and we don't have to rely on the model to correctly format presentation of the present labels for each pair.  Instead of letting the model choose, we build a specification: a short structured description of one sentence, listing the drugs it must mention and which pairs among them interact, and how. The specification is turned into a prompt, and the sentence the model writes is stored next to it. A second step finds each drug in the sentence, tags every pair, and copies each pair's label from the specification.

If the model writes a sentence that does not actually express the requested interaction, the label is wrong. The verifier in section 4 was first built to catch these (report §3.2, §4.6).

The version numbers count prompt revisions as new versions rather than generators that had a full generation. Versions 1 to 12 revised two earlier designs, single sentences with stated negatives and multi-sentence clinical vignettes, and only v6 and v11 were ever trained. v16 was a revision folded into v17 the same day. The five here, v13, v14, v15, v17, and v18, are the ones with multi-seed runs on the final evaluation sets (report Table 2).

Here is one sentence from v18, the final generator:

In [ ]:
rec = next(r for r in synth["v18"]
           if r["spec"] and r["spec"].get("positives") and r["n_entities"] >= 3)
spec = rec["spec"]
surface = dict(zip("ABCDEFGHIJKLMNOPQRSTUVWXYZ", spec.get("entities") or []))   # keys run A, B, C in list order

print("assertions ", spec.get("assertions"))
print("modifiers  ", spec.get("modifiers"))
print("variants   ", spec.get("variants"))
print("\ngold, as the specification states it")
for p in spec["positives"]:
    a, b = p["between"]
    print(f"  {surface.get(a, a)}  {p['label']}  {surface.get(b, b)}")
print("\nthe model wrote\n ", rec["sentence"], "\n")
pd.DataFrame([{"drug A": a, "drug B": b, "label": p["label"]}
              for p in rec["pairs"] for a, b in [pair_names(p["text"])]])

In [ ]:
from ddi.prompt import SYSTEM   # v18 reuses v17's system prompt

print("SYSTEM PROMPT\n")
print(SYSTEM)


In [ ]:

print("\nUSER PROMPT, rendered from the specification above\n")
print(spec["prompt"])

Above is an example of a spec after being rendered, along with the system prompt. The system prompt is the same for every sentence; the user prompt lists the drugs, the arrangement, and what the sentence must assert, but never a label. Pairs named in the specification get its labels and every other pair is NONE, which is how one generated sentence gives negative examples as well as positive ones.

### 2.1 What each generator produced

The simplest description of a dataset is how many drugs its sentences mention, how many pairs that gives, and what fraction of those pairs are positive.

In [ ]:
def describe(records):
    pairs = flatten(records)
    per = [len(r["pairs"]) for r in records]
    per.sort()
    return {"sentences": len(records),
            "pairs": len(pairs),
            "pairs/sentence mean": round(sum(per) / len(per), 2),
            "pairs/sentence median": per[len(per) // 2],
            "positive pairs": round(sum(p["label"] != "NONE" for p in pairs) / len(pairs), 3),
            "positive sentences": round(sum(sentence_label(r) == "POS" for r in records) / len(records), 3)}

pd.DataFrame({k: describe(v) for k, v in data.items()}).T

Looking at v13, almost every sentence has exactly one pair, with nine in ten of those pairs being positive. The generator did what it was asked, namely writing a sentence where two given drugs interact. Real sentences mention several drugs, and most of the pairs between them are NONE. For real text the mean is far above the median because a few sentences list dozens of drugs and produce hundreds of pairs.

### 2.2 The pair-count shortcut

The rule below fires positive based on the amount of entities (seperate drug mentions) in a sentence. Try editing it and see how well each generated dataset scores at 2 drugs, 3 drugs, etc.

In [ ]:
def my_rule(record):
    # edit this: it sees only how many drugs the sentence contains
    return "POS" if record["n_entities"] == 3 else "NONE"

def score_rule(rule, records):
    return sum(rule(r) == sentence_label(r) for r in records) / len(records)

{k: round(score_rule(my_rule, v), 3) for k, v in data.items()}

Firing on 2 entities scores well on v13 and badly on real text. A BERT classifier trained on v13 would find the same rule, because it is the simplest way to a low loss on that data. The table below splits the positive rate by number of drugs. Note the poor distributional coherence between v13 and the sentences from the human corpus.

In [ ]:
def positive_rate_by_entities(records):
    buckets = defaultdict(list)
    for r in records:
        for p in r["pairs"]:
            buckets[min(r["n_entities"], 6)].append(p["label"] != "NONE")
    return {k: round(sum(v) / len(v), 3) for k, v in sorted(buckets.items())}

pd.DataFrame({k: positive_rate_by_entities(v) for k, v in data.items()}) \
  .rename_axis("drugs (6 = six or more)").fillna("none written")

### 2.3 In detail: a label that can be read off the construction

v14 added drugs that take no part in the relation, which removed the pair-count shortcut. Precision went up but recall did not (scores in 2.4). To see why, we start from a single v17 sentence.

v17 built each sentence from one of nineteen named constructions, called frames, taken from the annotation guidelines. Below is an example:

In [ ]:
v17 = [r for r in synth["v17"] if r["spec"]]
focus = Counter(r["spec"]["frame"] for r in v17 if sentence_label(r) == "POS").most_common(1)[0][0]
one = next(r for r in v17 if r["spec"]["frame"] == focus and 3 <= r["n_entities"] <= 5)
keys = dict(zip("ABCDEFGHIJKLMNOPQRSTUVWXYZ", one["spec"].get("entities") or []))
fill = lambda v: re.sub(r"\{([A-Z])\}", lambda m: keys.get(m.group(1), m.group(0)), str(v))

print("frame:", focus)
print("\nthe specification asked for")
for k, v in one["spec"].get("says") or []:
    print(f"  {k}: {fill(v)}")
print("\nthe model wrote\n ", one["sentence"], "\n")
pd.DataFrame([{"drug A": a, "drug B": b, "label": p["label"]}
              for p in one["pairs"] for a, b in [pair_names(p["text"])]])

Looking at this example, it seems to produce a sentence that fits the description. Perfect! The specification asked for a relation, the model wrote it, and the pair carries the label the specification gave it. But the problem appears when we analyse multiple of these sentences. Here are the sentence labels of every sample sentence built from the same frame, then of every frame.

In [ ]:
same = [r for r in v17 if r["spec"]["frame"] == focus]
print(f"{len(same)} sample sentences use '{focus}':", dict(Counter(sentence_label(r) for r in same)), "\n")
pd.crosstab(pd.Series([r["spec"]["frame"] for r in v17], name="frame"),
            pd.Series([sentence_label(r) for r in v17], name="sentence label"))

Every row has a zero in one column, implying that each frame always gives the same sentence label, and so the frame alone can tell us the label. That matters because a classifier trained on these sentences can learn "this kind of sentence is positive" instead of learning to read whether two drugs interact.

The uncertainty coefficient puts a number on how much the construction gives away (report §3.5, Equation 2). With $Y$ the sentence label and $F$ the frame, the entropy of the label is $H(Y) = -\sum_y p(y)\log_2 p(y)$, and the entropy left once the frame is known is $H(Y \mid F) = \sum_f p(f)\,H(Y \mid F = f)$. The coefficient is the share of the label's uncertainty that the frame removes:

$$U(Y \mid F) = \frac{H(Y) - H(Y \mid F)}{H(Y)}$$

It is 1 when the frame decides the label and 0 when the frame says nothing about it. We first compute it by hand from the counts above (report Exercise 3.2):

In [ ]:
labels = Counter(sentence_label(r) for r in v17)
n = sum(labels.values())
H_Y = -sum(c / n * math.log2(c / n) for c in labels.values())

by_frame = defaultdict(Counter)
for r in v17:
    by_frame[r["spec"]["frame"]][sentence_label(r)] += 1
H_Y_given_F = sum(sum(c.values()) / n * -sum(k / sum(c.values()) * math.log2(k / sum(c.values()))
                                            for k in c.values() if k)
                  for c in by_frame.values())

print(f"H(Y)     = {H_Y:.3f} bits   from {dict(labels)}")
print(f"H(Y | F) = {H_Y_given_F:.3f} bits   every frame is pure, so each term is zero")
print(f"U(Y | F) = {(H_Y - H_Y_given_F) / H_Y:.3f}")

Using the same calculation as a function:

In [ ]:
def entropy(counts):
    n = sum(counts.values())
    return -sum(c / n * math.log2(c / n) for c in counts.values() if c)

def uncertainty_coefficient(pairs):
    """U(label | x) for (x, label) pairs: the share of the label's entropy that knowing x removes.
    1.0 means x decides the label, 0.0 means it says nothing about it."""
    joint = Counter(pairs)
    px, py = Counter(), Counter()
    for (x, y), c in joint.items():
        px[x] += c; py[y] += c
    n = sum(joint.values())
    mi = sum(c / n * math.log2((c / n) / ((px[x] / n) * (py[y] / n)))
             for (x, y), c in joint.items())
    return mi / entropy(py)

round(uncertainty_coefficient([(r["spec"]["frame"], sentence_label(r)) for r in v17]), 3)

To see what the number says about a generator, take 100 sentences, half built from a mechanism (a positive frame) construction and half from a denial (a negative frame), with half of all sentences positive.

In [ ]:
def toy(counts):   # {construction: (positive sentences, negative sentences)}
    pairs = [(f, lab) for f, (pos, neg) in counts.items()
             for lab, k in (("POS", pos), ("NONE", neg)) for _ in range(k)]
    return round(uncertainty_coefficient(pairs), 3)

{"every mechanism sentence positive, every denial negative": toy({"mechanism": (50, 0), "denial": (0, 50)}),
 "each construction four-fifths one label":                  toy({"mechanism": (40, 10), "denial": (10, 40)}),
 "each construction half and half":                          toy({"mechanism": (25, 25), "denial": (25, 25)})}

The first case is equivalent to what happened with v17. Since sentences made with the same frames tend to repeat words / phrasing / sentence structure the classifier can use this as a proxy to guess the label. Note that the last case goes too far to the other extreme. A generator whose denials are positive as often as they are negative is not writing realistic text either.

So this must be balanced to emulate real text. A sentence saying one drug raises the levels of another is usually positive and a denial is usually NONE, but not always, so the target is the corpus's own value (0.464, computed in 2.4), not zero. The number goes down when a design lets one construction produce both labels, for example by adding a second, negative assertion to a positive sentence. It goes up when constructions are tied to labels.

So how can we test whether the wording of the sentence can reveal the frame used? A technique we used was to hide every drug name and train the simplest possible text model to guess the frame. A bag of words counts which words and word pairs (bigrams) appear in a sentence and ignores their order. We train it on two thirds of the v17 sample and test it on the other third, three times over, so it is always scored on sentences it has not seen (k-fold cross-validation). The cell also lists the words the model relies on most for our example's frame, and which of them the example contains.

In [ ]:
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

vec = CountVectorizer(ngram_range=(1, 2), min_df=2)
X = vec.fit_transform([masked(r) for r in v17])
y = [r["spec"]["frame"] for r in v17]
clf = LogisticRegression(max_iter=3000).fit(X, y)
names = vec.get_feature_names_out()
top = [names[i] for i in np.argsort(clf.coef_[list(clf.classes_).index(focus)])[::-1][:12]]

print("sentence, masked\n ", masked(one), "\n")
print(f"strongest features for '{focus}':", ", ".join(top))
print("of those, present in this sentence:", ", ".join(w for w in top if w in masked(one).lower()) or "none")
acc = cross_val_score(LogisticRegression(max_iter=3000), X, y, cv=3).mean()
print(f"\nframe recovered from masked text: {acc:.3f} over {len(set(y))} frames (chance {1 / len(set(y)):.3f})")

The model names the right frame for about three sentences in four, where guessing at random would get one in nineteen. Looking at the words it relies on, our example contains "measured by", which the mechanism frame nearly always produces, and small words like "on" and "as" score highly because they come from the same template ("acts on", "as measured"). On the full 5,702-sentence dataset, with more examples of each frame to learn from, the same test gets 0.961.

So the words give away the frame, and the frame gives away the label. A BERT classifier can reach the label without ever deciding whether the two drugs interact. The most repeated phrases show where those words come from.

In [ ]:
def top_ngrams(records, n=4, k=10):
    c = Counter()
    for r in records:
        w = re.sub(r"[^\w\s]", " ", masked(r).lower()).split()
        c.update(" ".join(w[i:i + n]) for i in range(len(w) - n + 1))
    return c.most_common(k)

pd.DataFrame({"v17": [f"{g}  ({c})" for g, c in top_ngrams(synth["v17"])],
              "human": [f"{g}  ({c})" for g, c in top_ngrams(human)]})

v17 described every drug outside the relation with one of seven fixed phrases, and those phrases supply its most repeated 4-grams; real text's are drug enumerations. Its negative frames also avoided effect vocabulary, so words such as *caused* and *prolonged* mark positives and denial language marks NONE, neither of which holds in real text.

The corpus shows which constructions carry labels. 300 real sentences were classified by construction, once, by `gpt-oss-120b` at temperature 0, and committed in `runs/frames/`. For each construction the table gives how often it is the main clause of a sentence, how often it appears at all, and its positive rate in the corpus against v17.

v17 added 'hard negatives', ie. every drug outside the target positive relations is given a short description of it's role in the sentence, such as "part of the background regimen" or "used if the first choice is unsuitable". This was done to prevent the model from simply listing the extra drugs outside the main clause, which was making it easy for the classifier to identify negative labels from this listing pattern. However, as is often the case, fixing this problem lead to another. The vocabulary from the short instructions given for each 'hard negative' was then picked up by the classifier as an indication of label. We can see this in the most common 4-grams. In real text the most repeated 4-grams are lists of drugs. Because each fixed phrase is only ever attached to a drug outside the relation, it marks that drug's pairs as NONE without any pharmacology.

To see which constructions really carry labels, we compare with the corpus. 300 real sentences were classified by construction once, by `gpt-oss-120b` at temperature 0, and saved in `runs/frames/`. For each construction the table gives how often it is the main clause of a sentence, how often it appears at all, and its positive rate in the corpus against v17.

In [ ]:
import json
from pathlib import Path

gold = [json.loads(l) for l in (lib.REPO / "runs/frames/gold_frames_n300_seed0.jsonl").read_text().splitlines() if l]
gold = [g for g in gold if not g.get("error") and g.get("confident", True)]

primary, present, corpus_pos = Counter(), Counter(), defaultdict(list)
for g in gold:
    primary[g["primary"]] += 1
    for f in {g["primary"], *(g.get("also_present") or [])}:
        present[f] += 1
        corpus_pos[f].append(g["has_pos"])

v17_pos = defaultdict(list)
for r in synth["v17"]:
    if r["spec"]:
        v17_pos[r["spec"]["frame"]].append(sentence_label(r) == "POS")

rows = []
for f in sorted(present, key=lambda f: -present[f]):
    c = sum(corpus_pos[f]) / len(corpus_pos[f])
    s = sum(v17_pos[f]) / len(v17_pos[f]) if v17_pos[f] else None
    rows.append({"construction": f, "main clause": primary[f], "present in": present[f],
                 "corpus positive": round(c, 3),
                 "v17 positive": None if s is None else round(s, 3),
                 "gap": None if s is None else round(abs(s - c), 3)})
pd.DataFrame(rows)

Appositive and coordinate constructions appear in real sentences but are never the main clause, because they package an assertion rather than make one. v17 treated them as frames of their own and made appositive sentences positive every time, where the corpus rate is about four in ten. Note that every v17 frame is 0 or 1 in the v17 column, while the corpus column is spread in between. That spread is what v18 has to reproduce.

### 2.4 The fix: two axes

To fix this, in v18 assertions bind a pair of drugs and carry a label, and a sentence draws one or two of them, the second usually negative. Modifiers carry no label and are drawn independently, examples including headings, repeated mentions, dose details, appositives. Because a positive assertion can now appear next to a negative one, and any assertion can appear with any modifier, no single part of a spec decides the sentence's label. The cell below builds 3000 specs and show one of them.

In [ ]:
from ddi.vocab import build_vocab
from ddi import prompt_v18 as p18

vocab = build_vocab()
specs = p18.make_specs(3000, vocab=vocab, seed=0)

for s in specs[:8]:
    print(f"{'+'.join(s['assertions']):<28} {', '.join(s['modifiers']) or '-':<42} "
          f"{'positive' if s['positives'] else 'NONE'}")

Below we calculated the same coefficient for v17's frames, v18's assertion sets, and the corpus's constructions. The closer we can get to the corpus uncertainty coefficient the better, since real constructions carry some information about the label, and a generator with a uncertainty coefficient of zero would be writing denials as often positive as negative.

In [ ]:
from ddi import prompt as p17

v17_specs = p17.make_specs(3000, vocab=vocab, seed=0)
pd.Series({
    "v17  frame": uncertainty_coefficient([(x["frame"], bool(x["positives"])) for x in v17_specs]),
    "v18  assertion set": uncertainty_coefficient(
        [("+".join(sorted(x["assertions"])), bool(x["positives"])) for x in specs]),
    "corpus  construction": uncertainty_coefficient([(g["primary"], g["has_pos"]) for g in gold]),
}).round(3)

v18's coefficient is 0.458 against the corpus's 0.464. In practice this change took F1 from 0.391 to 0.486 [dev], the gain coming almost solely from recell, 0.403 to 0.628, with precision nearly unchanged. This could indicate the classifier stopped relying on a cue that real text does not contain. The micro F1 scores [dev] of every version:

In [ ]:
pd.DataFrame(R["trajectory"]).T[["f1", "sd", "p", "r", "n_seeds"]]

### 2.5 Extending the generator

Each of v18's seventeen assertions is a small function. It is given the sentence's drugs (each with a key: A, B, C…) and a random number generator, and returns four things:

- `kind`, which decides which stylistic options the renderer can sample;
- `says`, what the sentence should say / assert;
- `positives`, the pairs it makes positive and with which label;
- `focus`, the drugs it is about.

In the inventory each assertion also has `n`, how many drugs it needs, and `w`, how often it is drawn compared with the others. Here is the existing `mechanism` assertion run on four made-up drugs (rerun to see multiple examples):

Mechanism:

In [ ]:
ents = [{"key": k, "surface": s} for k, s in zip("ABCD", ["warfarin", "rifampicin", "aspirin", "omeprazole"])]
print({k: v for k, v in p18.ASSERTIONS["mechanism"].items() if k != "f"})
p18.ASSERTIONS["mechanism"]["f"](ents, random.Random())

`mechanism` picks two drugs, one acting on the other, and samples what changes from a list of pharmacokinetic sites (liver metabolism, absorption, kidney clearance, protein binding, transport, stomach acidity and distribution), along with a direction and how it was measured. Protein binding is already one of those sites, so some mechanism sentences are already about one drug displacing another. A dedicated assertion is how we would make one kind of sentence its own construction, with fixed wording and its own weight.

Advise:

In [ ]:
ents = [{"key": k, "surface": s} for k, s in zip("ABCD", ["warfarin", "rifampicin", "aspirin", "omeprazole"])]
print({k: v for k, v in p18.ASSERTIONS["advise"].items() if k != "f"})
p18.ASSERTIONS["advise"]["f"](ents, random.Random())

Effect:

In [ ]:
ents = [{"key": k, "surface": s} for k, s in zip("ABCD", ["warfarin", "rifampicin", "aspirin", "omeprazole"])]
print({k: v for k, v in p18.ASSERTIONS["effect"].items() if k != "f"})
p18.ASSERTIONS["effect"]["f"](ents, random.Random())

Interaction:

In [ ]:
ents = [{"key": k, "surface": s} for k, s in zip("ABCD", ["warfarin", "rifampicin", "aspirin", "omeprazole"])]
print({k: v for k, v in p18.ASSERTIONS["int"].items() if k != "f"})
p18.ASSERTIONS["int"]["f"](ents, random.Random())

`f_displacement` below has the same shape as the above , but "what changes" is always binding to plasma proteins and "which way" is always displaced. Its `kind` has to be one of these, or `make_specs` raises a KeyError:

In [ ]:
sorted(p17.AXES_FOR_KIND)

We register it with `n` = 2 and `w` = 0.06 and build 3,000 specs, without generating anything:

In [ ]:
def f_displacement(ents, rng):
    a, b = rng.sample([e["key"] for e in ents], 2)
    return {"kind": "positive",
            "says": [("acts", f"{{{a}}}"), ("acted on", f"{{{b}}}"),
                     ("what changes", "binding to plasma proteins"), ("which way", "displaced")],
            "positives": [(a, b, "MECHANISM")],
            "focus": [a, b]}

saved = dict(p18.ASSERTIONS)
try:
    p18.ASSERTIONS["displacement"] = {"n": 2, "w": 0.06, "f": f_displacement}
    extended = p18.make_specs(3000, vocab=vocab, seed=0)
finally:
    p18.ASSERTIONS.clear(); p18.ASSERTIONS.update(saved)

drawn = [x for x in extended if "displacement" in x["assertions"]]
alone = [x for x in drawn if len(x["assertions"]) == 1]
print(f"drawn in {len(drawn)} of {len(extended)} specs")
print(f"sentence positive rate when it is the only assertion: "
      f"{sum(bool(x['positives']) for x in alone) / len(alone):.3f}")

On its own, the new assertion gives a positive sentence about 76% of the time. The split below shows where the other quarter comes from (rerun to see mutiple examples). Pay attention to the 'acted on' field. A random non drug entity is chosen

In [ ]:
print(pd.crosstab(pd.Series(["unlisted" in x["variants"] for x in alone], name="unlisted variant"),
                  pd.Series(["positive" if x["positives"] else "NONE" for x in alone], name="sentence label")))

ex = random.choice([x for x in alone if "unlisted" in x["variants"]])
print("\none of the NONE specs, rendered as the model would see it:\n")
print(p18.render(ex))

key = lambda x: "+".join(sorted(x["assertions"]))
print("\nuncertainty coefficient of assertion sets, before and after adding it:",
      round(uncertainty_coefficient([(key(x), bool(x["positives"])) for x in specs]), 3),
      round(uncertainty_coefficient([(key(x), bool(x["positives"])) for x in extended]), 3))

It's only NONE specs which have the `unlisted` variant. The variant keeps the assertion's wording, a drug displacing another from plasma proteins, but retargets it: "acted on" becomes "an endogenous substance", and the prompt tells the model not to name any listed drug as the thing affected. The sentence reads like a mechanism statement, yet none of its drug pairs interact, so it is a hard negative. v18 applies this to about a quarter of positive assertions, and a new assertion inherits it automatically, so its wording never guarantees a positive label. The uncertainty coefficient from 2.3 confirms it: 0.458 with or without the new assertion. We know it cannot become a new shortcut before paying for any generation.

Let's do the same check for some other v18 assertions. `show` tabulates every spec containing an assertion, split by whether it is alone (and with which variant) or next to a second assertion, then renders one example. Here we inspect the `effect` assertion (rerun to see multiple examples and feel free to change the variant selected). 

In [ ]:
def context(x, name):
    if len(x["assertions"]) == 1:
        return "alone" + (f", {x['variants'][0]}" if x["variants"] else "")
    other = next(a for a in x["assertions"] if a != name)
    return f"with {other}" if other in ("effect", "mechanism", "advise") else "with another assertion"

def show(name, pick):
    has = [x for x in specs if name in x["assertions"]]
    print(pd.crosstab(pd.Series([context(x, name) for x in has], name=name),
                      pd.Series(["positive" if x["positives"] else "NONE" for x in has], name="sentence label")))
    ex = random.Random().choice([x for x in has if pick(x)])
    print(f"\nassertions {ex['assertions']}   variants {ex['variants']}   positives {ex['positives']}\n")
    print(p18.render(ex))

show("effect", lambda x: x["variants"] == ["list_target"] and len(x["assertions"]) == 1)

Alone, `effect` behaves like the new assertion: `unlisted` makes it NONE. `list_target` goes the other way, turning one assertion into a positive pair with each drug in a list (note the multiple positive pairs), which is how a single sentence gives several EFFECT pairs.

In [ ]:
show("denial", lambda x: len(x["assertions"]) == 2 and bool(x["positives"]))

A denial on its own is always NONE. It only ends up in a positive sentence next to a second assertion that makes a different pair positive, as in the example, where one pair has a synergistic effect and the other shows no measurable difference. This is how v18 puts denials into positive sentences, as real text does: in the corpus table in 2.3, sentences with a denial are positive about one time in five.

In [ ]:
show("contradictory", lambda x: len(x["assertions"]) == 1 and bool(x["positives"]))

`contradictory` says an interaction has not been reported for one drug but that the advice still applies with another. The advice is what gets annotated, so the pair is ADVISE rather than NONE, a convention a generator has to be told about.

## 3. Labelling real sentences

The other use of the model keeps real sentences and their marked drugs and asks only for the labels. The annotator sees eight pairs per request, with a simplifed version of the guideline conventions in its prompt. Here is one sentence from the labelled sample, with the gold label and the model's label for every pair.

In [ ]:
lab = lib.load_labelled()   # built on the pod by make_samples.py --labelled

def fp_rejected(p):
    return p["llm"] != "NONE" and p["gold"] == "NONE" and p["verdict"] == "rejected"

pool = [r for r in lab if any(fp_rejected(p) for p in r["pairs"]) and len(r["pairs"]) >= 3]
story = min(pool, key=lambda r: len(r["pairs"]))
print(story["sentence"], "\n")
pd.DataFrame([{"drug A": a, "drug B": b, "gold": p["gold"], "model": p["llm"]}
              for p in story["pairs"] for a, b in [pair_names(p["text"])]])

At least one pair here is positive to the model and NONE in the gold labels. Across the sample the pattern is the same: the model finds most real interactions and adds interactions the annotators did not mark. The label-level scores, the model's labels scored against gold with the task metric:

In [ ]:
def prf(pairs, key):
    tp = sum(1 for p in pairs if p[key] != "NONE" and p[key] == p["gold"])
    fp = sum(1 for p in pairs if p[key] != "NONE" and p[key] != p["gold"])
    fn = sum(1 for p in pairs if p["gold"] != "NONE" and p[key] != p["gold"])
    P, Rc = tp / max(tp + fp, 1), tp / max(tp + fn, 1)
    return {"precision": round(P, 3), "recall": round(Rc, 3),
            "F1": round(2 * P * Rc / max(P + Rc, 1e-9), 3), "positives": tp + fp}

pairs = [p for r in lab for p in r["pairs"]]
print("gold positives:", sum(p["gold"] != "NONE" for p in pairs), "of", len(pairs), "pairs")
prf(pairs, "llm")

On the full labelled pool the annotator reached precision 0.623 and recall 0.881 against gold [dev]. Trained on, those labels cost 0.126 F1 against the same sentences with human labels [test]:

In [ ]:
T = R["test"]
readable = {"test-bfull-none": "human labels", "test-b0-llm": "model labels", "test-b0-llmf": "model labels, checked"}
pd.DataFrame({readable[k]: T[k] for k in ["test-bfull-none", "test-b0-llm"]}).T[["f1", "sd", "p", "r", "n_seeds"]]

## 4. In detail: checking the model's labels

The model's errors are mostly false positives, so the check asks about positives only. A second call, the verifier, receives the sentence with every drug mention numbered and one question per pair: would the guidelines annotate an interaction between these two mentions? A pair the verifier rejects is demoted to NONE. This is the exact question it was sent for the sentence above.

In [ ]:
from ddi.verify_binary import SYSTEM, build_batches, render

asked = [{"sent_id": story["sent_id"], "text": p["text"], "label": p["llm"]}
         for p in story["pairs"] if p["llm"] != "NONE"]
print("\n".join(SYSTEM.splitlines()[:9]), "\n  ...\n")
print(render(build_batches(asked)[0]))

Mentions are numbered rather than named because the same drug named twice is two mentions, and the guidelines let only one of them take part in an interaction. Only the model's positive pairs are asked about, so the numbering counts the mentions in those pairs. The verdicts, and the labels after demotion:

In [ ]:
pd.DataFrame([{"drug A": a, "drug B": b, "gold": p["gold"], "model": p["llm"],
               "verifier": p["verdict"], "after": p["final"]}
              for p in story["pairs"] for a, b in [pair_names(p["text"])]])

In this sentence the verifier rejects a pair that is NONE in gold, and demotion corrects it. Whether that holds across the sample decides whether the check works: the rejected positives should be NONE in gold far more often than the accepted ones.

In [ ]:
llm_pos = [p for p in pairs if p["llm"] != "NONE"]
def gold_none(ps):
    k = sum(p["gold"] == "NONE" for p in ps)
    return f"{k:>4} of {len(ps):<4} ({k / max(len(ps), 1):.0%})"
print("model positives that are NONE in gold  ", gold_none(llm_pos))
print("  rejected by the verifier             ", gold_none([p for p in llm_pos if p["verdict"] == "rejected"]))
print("  accepted by the verifier             ", gold_none([p for p in llm_pos if p["verdict"] == "accepted"]))
print("  not asked, the request failed        ", gold_none([p for p in llm_pos if p["verdict"] == "not asked"]))

NOTE: state the result once `labelled.jsonl` exists. Give the share of rejected positives that are NONE in gold against the share among accepted ones, and say in one sentence whether the verifier is removing false positives or rejecting at random.

The check also has a cost: a rejected pair that is positive in gold loses a true interaction.

In [ ]:
cost = [(r, p) for r in lab for p in r["pairs"] if p["verdict"] == "rejected" and p["gold"] != "NONE"]
print(f"{len(cost)} rejected pairs are positive in gold")
if cost:
    r, p = cost[0]
    a, b = pair_names(p["text"])
    print(f"\n{r['sentence']}\n\n{a} and {b}: gold {p['gold']}, model {p['llm']}, demoted to NONE")

NOTE: describe the example above in one sentence, and what kind of pair the verifier wrongly rejects.

Scored against gold before and after demotion:

In [ ]:
pd.DataFrame({"before checking": prf(pairs, "llm"), "after checking": prf(pairs, "final")})

On the test set, the classifier trained on checked labels moved from precision 0.578 and recall 0.824 to 0.676 and 0.790 [test], and checking recovered 39% of the label cost from section 3:

In [ ]:
pd.DataFrame({readable[k]: T[k] for k in ["test-bfull-none", "test-b0-llmf", "test-b0-llm"]}).T[["f1", "sd", "p", "r", "n_seeds"]]

## 5. Spending annotation

With some human-labelled sentences and more without labels, the question is what to add. The grid fixed a human budget of 0 to 1,000 sentences and added nothing, model labels on the remaining sentences, checked model labels, generated v18 text, or both. These are validation scores. Every arm trained for at least 500 optimisation steps; with a fixed three epochs, the smallest human-only arms collapse to predicting NONE for every pair, which makes anything added to them look far more valuable than it is.

In [ ]:
grid = R["grid"]
adds = [("none", "human only"), ("llm", "+ LLM labels"), ("llmf", "+ LLM labels, verified"),
        ("synth", "+ generated v18"), ("both", "+ LLM labels and v18")]
budgets = [0, 100, 250, 500, 1000]
tab = pd.DataFrame({label: {b: grid[str(b)].get(a, {}).get("f1") for b in budgets}
                    for a, label in adds}).rename_axis("human-labelled sentences")
ax = tab.plot(marker="o", figsize=(7, 4), ylabel="validation micro-F1", xticks=budgets)
ax.set_xticklabels(budgets)
tab.round(3).fillna("-")   # no human-only arm exists at a budget of 0

The configurations validation chose, scored once on the test set:

In [ ]:
order = ["test-b0-synth", "test-b0-llm", "test-b0-llmf", "test-b1000-none", "test-b1000-llmf", "test-bfull-none"]
names = {"test-b0-synth": "generated v18", "test-b0-llm": "model labels",
         "test-b0-llmf": "model labels, checked", "test-b1000-none": "1,000 human only",
         "test-b1000-llmf": "1,000 human + checked model labels", "test-bfull-none": "all human"}
tt = pd.DataFrame({names[k]: {"human sentences": int(T[k]["human_sentences"]), "F1": T[k]["f1"], "sd": T[k]["sd"],
                              "P": T[k]["p"], "R": T[k]["r"]} for k in order}).T
tt["share of all human"] = (tt["F1"] / T["test-bfull-none"]["f1"]).round(3)
tt

With no human labels, checked model labels reach 90% of full human supervision, within 0.02 of 1,000 human-labelled sentences on their own; with 1,000 human sentences, 42% of the set, they reach 96%. Generated text reaches 65%, and added to model labels it lowered F1 at every budget on validation. The write-up gives these results with their uncertainties.

## 6. Applying this to another task

The machinery carries over unchanged: specification, render and generate; the two-stage split between stored model output and deterministic instance building; the verifier's numbered-mention question; and the diagnostics in section 2, which need only a specification with a construction field and a sample of real sentences classified the same way. What is specific to DDI is the label set, the guideline conventions in the annotator's and verifier's prompts, the drug vocabulary, and the pair structure.

Checks worth running before generating anything:

- Compare model labels on real text with generated text on the same evaluation set before improving either.
- Build specifications and measure each construction's label purity and the uncertainty coefficient against a classified corpus sample; both are free.
- Train every comparison to a minimum number of optimisation steps, not a fixed number of epochs.
- Before trusting a filter, count what it removes by category.
- Choose on a held-out split, freeze the choice, and test once.

## Optional: train on a sample

Needs a GPU. Reproduces the shape of the pair-count result on the 400-sentence samples, not the logged figures.

In [ ]:
# dev = human_sentences("dev")
# for v in ["v13", "v14", "v18"]:
#     print(v, lib.train_eval(synth[v], dev, seeds=(0,)))